# Manifest Alchemy — LLM Fine-Tune (Llama 3.1 8B)
**Free training on Google Colab T4 GPU using Unsloth + QLoRA**

## Before running:
1. Upload your `dataset.jsonl` to this Colab session (Files panel → Upload)
2. Runtime → Change runtime type → **T4 GPU**
3. Run all cells in order — takes ~2 hours
4. Download `manifest-coach-lora.gguf` from the Files panel when done

In [ ]:
# Cell 1 — Check GPU
import subprocess
r = subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv,noheader'],
                   capture_output=True, text=True)
print('GPU:', r.stdout.strip())

In [ ]:
# Cell 2 — Install Unsloth (takes ~3 min)
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git" -q
!pip install "bitsandbytes>=0.43.0" trl transformers datasets accelerate -q
print('All packages installed.')

In [ ]:
# Cell 3 — Load base model (downloads ~5 GB, takes ~5 min)
from unsloth import FastLanguageModel
import torch

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name='unsloth/Meta-Llama-3.1-8B-Instruct-bnb-4bit',
    max_seq_length=2048,
    dtype=torch.float16,
    load_in_4bit=True,
)
print('Base model loaded.')

In [ ]:
# Cell 4 — Apply LoRA adapters
model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj',
                    'gate_proj', 'up_proj', 'down_proj'],
    lora_alpha=16,
    lora_dropout=0,
    bias='none',
    use_gradient_checkpointing='unsloth',
    random_state=42,
)
print('LoRA adapters applied.')

In [ ]:
# Cell 5 — Load dataset
# Make sure dataset.jsonl is uploaded to this Colab session first!
from datasets import load_dataset

dataset = load_dataset('json', data_files='/content/dataset.jsonl', split='train')

def format_messages(example):
    return {
        'text': tokenizer.apply_chat_template(
            example['messages'],
            tokenize=False,
            add_generation_prompt=False
        )
    }

dataset = dataset.map(format_messages, remove_columns=dataset.column_names)
print(f'Dataset loaded: {len(dataset)} examples')
print('Sample:', dataset[0]['text'][:200])

In [ ]:
# Cell 6 — Train (~2 hours on T4)
from trl import SFTTrainer
from transformers import TrainingArguments

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=dataset,
    dataset_text_field='text',
    max_seq_length=2048,
    dataset_num_proc=2,
    packing=False,
    args=TrainingArguments(
        per_device_train_batch_size=2,
        gradient_accumulation_steps=4,
        warmup_steps=20,
        num_train_epochs=2,
        learning_rate=2e-4,
        fp16=True,
        logging_steps=10,
        output_dir='/content/outputs',
        optim='adamw_8bit',
        weight_decay=0.01,
        lr_scheduler_type='cosine',
        seed=42,
        save_steps=200,
        save_total_limit=2,
    ),
)

stats = trainer.train()
print(f'Done! Loss: {stats.training_loss:.4f}')

In [ ]:
# Cell 7 — Test the model before saving
FastLanguageModel.for_inference(model)

SYSTEM = """You are Manifest Alchemy AI — an intelligent manifestation architect that blends magic, logic, alchemy, and algorithm to help users create their manifestations into reality.
Ask only ONE question per response. Have a natural, conversational flow."""

test_inputs = [
    'I want to manifest a new car.',
    'I want to attract my soulmate.',
    'I feel stuck and don\'t know what I want.',
]

for msg in test_inputs:
    inputs = tokenizer.apply_chat_template(
        [{'role': 'system', 'content': SYSTEM}, {'role': 'user', 'content': msg}],
        tokenize=True, add_generation_prompt=True, return_tensors='pt'
    ).to('cuda')
    out = model.generate(input_ids=inputs, max_new_tokens=150, temperature=0.8,
                         repetition_penalty=1.1)
    decoded = tokenizer.decode(out[0][inputs.shape[1]:], skip_special_tokens=True)
    print(f'User: {msg}')
    print(f'AI: {decoded}')
    print('---')

In [ ]:
# Cell 8 — Save LoRA weights
model.save_pretrained('/content/manifest-coach-lora')
tokenizer.save_pretrained('/content/manifest-coach-lora')
print('LoRA weights saved to /content/manifest-coach-lora')

In [ ]:
# Cell 9 — Convert to GGUF (for Ollama on your local machine)
# This lets you run the model locally without any API cost
model.save_pretrained_gguf(
    '/content/manifest-coach-gguf',
    tokenizer,
    quantization_method='q4_k_m'  # Best quality/size tradeoff for 6 GB RAM
)

import os, glob
gguf_files = glob.glob('/content/manifest-coach-gguf/*.gguf')
for f in gguf_files:
    size_gb = os.path.getsize(f) / 1e9
    print(f'GGUF file: {f}  ({size_gb:.1f} GB)')

print()
print('Download this .gguf file from the Files panel (left sidebar).')
print('Then run: ollama create manifest-coach -f training/Modelfile')

In [ ]:
# Cell 10 — (Optional) Push to HuggingFace Hub for Replicate deployment
# Only run this if you want to deploy to Replicate in production
# Get a free token at huggingface.co/settings/tokens

# from huggingface_hub import login
# login(token='your_hf_token')
# model.push_to_hub('your-username/manifest-coach-llama3.1-8b-lora')
# tokenizer.push_to_hub('your-username/manifest-coach-llama3.1-8b-lora')
print('Uncomment above lines and add your HuggingFace token to push to Hub.')